# Character-Consistent Generation — SDXL + LoRA (Colab)

Runs **end-to-end on a free Colab T4**. Order is deliberate:
1. **Baseline generation** with base SDXL  → real images fast
2. **Eval harness**  → real metrics (CLIP-T, self-consistency; identity if you add references)
3. **(Stretch) LoRA training** for character consistency
4. **A/B**: regenerate with the LoRA and compare metrics

> **Runtime → Change runtime type → T4 GPU** before running anything.
> Sections 1–2 give you "something working" in ~20 min. Section 3 is optional and slower.

## 0. Check GPU + install

In [ ]:
!nvidia-smi -L

In [ ]:
%pip -q install "diffusers>=0.27" "transformers>=4.38" accelerate peft safetensors pillow
# For LoRA training later (section 3):
%pip -q install bitsandbytes
print("installed")

## 1. Baseline generation with base SDXL

Produces the "before" images. On a T4 we use model CPU offload to fit 1024px SDXL.
A handful of images takes a few minutes.

In [ ]:
import torch, os
from diffusers import StableDiffusionXLPipeline, AutoencoderKL
from PIL import Image

PROMPTS = [
    "a portrait of sks person, studio lighting, high detail",
    "sks person as an astronaut floating in space, cinematic",
    "sks person sitting in a cozy coffee shop, warm morning light",
    "sks person hiking a mountain trail at sunrise",
    "sks person in a neon-lit cyberpunk city at night, rain",
    "sks person wearing medieval knight's armor in a castle",
]
SEED, STEPS, GUIDANCE = 42, 30, 7.0

def load_pipe(lora_dir=None):
    vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16)
    pipe = StableDiffusionXLPipeline.from_pretrained(
        "stabilityai/stable-diffusion-xl-base-1.0",
        vae=vae, torch_dtype=torch.float16, variant="fp16", use_safetensors=True)
    if lora_dir:
        pipe.load_lora_weights(lora_dir)
    pipe.enable_model_cpu_offload()   # T4-friendly; do NOT also call .to("cuda")
    pipe.set_progress_bar_config(disable=True)
    return pipe

def make_grid(images, cols=3):
    rows = (len(images) + cols - 1)//cols
    w, h = images[0].size
    grid = Image.new("RGB", (cols*w, rows*h), "white")
    for i, im in enumerate(images):
        grid.paste(im, ((i%cols)*w, (i//cols)*h))
    return grid

def generate(pipe, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    gen = torch.Generator(device="cuda").manual_seed(SEED)
    imgs = []
    for i, p in enumerate(PROMPTS):
        print(f"[{i+1}/{len(PROMPTS)}] {p}")
        img = pipe(prompt=p, num_inference_steps=STEPS,
                   guidance_scale=GUIDANCE, generator=gen).images[0]
        img.save(f"{out_dir}/gen_{i:02d}.png"); imgs.append(img)
    grid = make_grid(imgs); grid.save(f"{out_dir}/grid.png")
    return imgs, grid

base_pipe = load_pipe()
base_imgs, base_grid = generate(base_pipe, "/content/outputs/base")
base_grid

## 2. Eval harness

- **CLIP-T** — does each image match its prompt? (prompt fidelity)
- **Self-consistency** — do generations look like the *same* character across scenes?
- **CLIP-I / DINO identity** — only if you upload reference photos to
  `/content/data/my_character/` (skipped automatically if none).

In [ ]:
import torch, itertools, glob
from PIL import Image
from transformers import CLIPModel, CLIPProcessor, AutoModel, AutoImageProcessor

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

class CLIPScorer:
    def __init__(self, name="openai/clip-vit-large-patch14"):
        self.m = CLIPModel.from_pretrained(name).to(DEVICE).eval()
        self.p = CLIPProcessor.from_pretrained(name)
    @torch.no_grad()
    def img(self, images):
        x = self.p(images=list(images), return_tensors="pt").to(DEVICE)
        return torch.nn.functional.normalize(self.m.get_image_features(**x), dim=-1)
    @torch.no_grad()
    def txt(self, texts):
        x = self.p(text=list(texts), return_tensors="pt", padding=True, truncation=True).to(DEVICE)
        return torch.nn.functional.normalize(self.m.get_text_features(**x), dim=-1)

class DINOScorer:
    def __init__(self, name="facebook/dino-vitb16"):
        self.m = AutoModel.from_pretrained(name).to(DEVICE).eval()
        self.p = AutoImageProcessor.from_pretrained(name)
    @torch.no_grad()
    def img(self, images):
        x = self.p(images=list(images), return_tensors="pt").to(DEVICE)
        cls = self.m(**x).last_hidden_state[:, 0]
        return torch.nn.functional.normalize(cls, dim=-1)

def load_imgs(folder):
    paths = []
    for e in ("*.png","*.jpg","*.jpeg","*.webp"):
        paths += sorted(glob.glob(f"{folder}/{e}"))
    paths = [p for p in paths if "grid" not in p.lower()]
    return [Image.open(p).convert("RGB") for p in paths]

def clip_t(clip, imgs, prompts):
    return float((clip.img(imgs) * clip.txt(prompts)).sum(-1).mean())

def identity(scorer, gen, ref):
    return float((scorer.img(gen) @ scorer.img(ref).T).mean())

def self_consistency(scorer, gen):
    e = scorer.img(gen); n = len(e)
    s = [float(torch.dot(e[i], e[j])) for i, j in itertools.combinations(range(n), 2)]
    return sum(s)/len(s)

def evaluate(gen_dir, ref_dir="/content/data/my_character", label=""):
    clip, dino = CLIPScorer(), DINOScorer()
    gen = load_imgs(gen_dir)
    ref = load_imgs(ref_dir) if os.path.isdir(ref_dir) else []
    r = {
        "CLIP-T (prompt fidelity)": round(clip_t(clip, gen, PROMPTS[:len(gen)]), 4),
        "Self-consistency (CLIP)": round(self_consistency(clip, gen), 4),
        "Self-consistency (DINO)": round(self_consistency(dino, gen), 4),
        "CLIP-I identity": round(identity(clip, gen, ref), 4) if ref else "n/a (add refs)",
        "DINO identity": round(identity(dino, gen, ref), 4) if ref else "n/a (add refs)",
    }
    print(f"=== {label} ===")
    for k, v in r.items(): print(f"  {k:<26} {v}")
    return r

base_metrics = evaluate("/content/outputs/base", label="Base SDXL")

## 3. (Stretch) Train a character LoRA

**Only do this if sections 1–2 work and you have time.** It's the slow part.

**First:** upload 8–15 photos of ONE character to `/content/data/my_character/`
(left sidebar → Files → create the folder → upload). Varied pose/lighting, same identity.

T4 memory is tight for SDXL — we train at 768px with 8-bit Adam + gradient
checkpointing. If you hit OOM, drop `--resolution` to 512 or `--rank` to 8.

In [ ]:
# grab the official diffusers training script
!wget -q -O train_dreambooth_lora_sdxl.py https://raw.githubusercontent.com/huggingface/diffusers/main/examples/dreambooth/train_dreambooth_lora_sdxl.py
!accelerate config default
print("ready")

In [ ]:
!accelerate launch train_dreambooth_lora_sdxl.py \
  --pretrained_model_name_or_path="stabilityai/stable-diffusion-xl-base-1.0" \
  --pretrained_vae_model_name_or_path="madebyollin/sdxl-vae-fp16-fix" \
  --instance_data_dir="/content/data/my_character" \
  --output_dir="/content/outputs/lora" \
  --instance_prompt="a photo of sks person" \
  --resolution=768 --train_batch_size=1 --gradient_accumulation_steps=4 \
  --gradient_checkpointing --learning_rate=1e-4 --lr_scheduler="constant" \
  --rank=16 --max_train_steps=800 --checkpointing_steps=400 --seed=42 \
  --mixed_precision="fp16" --use_8bit_adam \
  --enable_xformers_memory_efficient_attention

## 4. A/B: generate with the LoRA + compare

The before/after table is your headline result for the interview — it shows the
fine-tune *measurably* moved identity/consistency, not just vibes.

In [ ]:
del base_pipe
torch.cuda.empty_cache()
lora_pipe = load_pipe(lora_dir="/content/outputs/lora")
lora_imgs, lora_grid = generate(lora_pipe, "/content/outputs/lora_gen")
lora_metrics = evaluate("/content/outputs/lora_gen", label="LoRA fine-tuned")

print("\n=== BEFORE vs AFTER ===")
for k in base_metrics:
    print(f"  {k:<26} base={base_metrics[k]}  ->  lora={lora_metrics[k]}")
lora_grid